In [1]:
import pygsti
import itertools
from pygsti.extras import ml
import numpy as np
from pygsti.circuits import Circuit
from pygsti.baseobjs import Label as L, Basis, QubitSpace
from pygsti.processors.processorspec import QubitProcessorSpec as QPS
from matplotlib import pyplot as plt
from pygsti.algorithms.randomcircuit import create_random_circuit as random_circuit
import tensorflow as tf
%load_ext autoreload
%autoreload 2

2026-02-02 13:00:05.980091: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
circuits = [pygsti.circuits.Circuit('[Gxpi2:0][Gypi2:0]@(0,1)'),
            #pygsti.circuits.Circuit('[Gxpi2:0][Gypi2:0]@(0,1)'
           ]

In [3]:
num_qubits = 2

gate_names = ['Gxpi2', 'Gypi2', 'Gcphase']

availability= {'Gcphase': [(0,1),]}

qubit_labels = [i for i in range(num_qubits)]

pspec = QPS(num_qubits = num_qubits, qubit_labels=qubit_labels,
            gate_names=gate_names, availability=availability)


circuits = [pygsti.circuits.Circuit('[Gxpi2:0][Gypi2:0]@(0,1)'),
            pygsti.circuits.Circuit('[Gxpi2:0][Gxpi2:0]@(0,1)')
           ]

modelled_error_generators = [('H',('XI',)), ('S',('XI',)), ('S',('YI',)), ('S',('ZI',)), ('S',('IX',))]

tensors = ml.encoding.error_generator_tensors(circuits, modelled_error_generators, pspec, alpha_representation='concise')
indices = tensors['indices'] # Not needed by the QPANN but are computed to compute the alphas and so still returned
signs = tensors['signs']  # Not needed by the QPANN but are computed to compute the alphas and so still returned
probabilities = tensors['probabilities'] 
alphas = tensors['alphas']

circuit_index = 0

print(probabilities[circuit_index])
#bit_string_index = 0
layer_index = 1
# Rows are the different bit strings, columns the different modelled error gens.
print(alphas[circuit_index, :, layer_index])

100%|██████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.12s/it]

[0.5 0.  0.5 0. ]
[[-1.   0.   0.   0.  -0.5]
 [ 0.   0.   0.   0.   0.5]
 [ 1.   0.   0.   0.  -0.5]
 [ 0.   0.   0.   0.   0.5]]


[0.5 0.  0.5 0. ]
[[-1.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.5]
 [ 1.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.5]]


[[-1.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.5]
 [ 1.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.5]]


In [ ]:
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
x = [circuits_tensor, alphas, probabilities]
qpann = ml.qpanns.QPANN(encoder.length, modelled_error_generators, snipper, probability_computation='concise')

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
early_stopping = tf.keras.callbacks.EarlyStopping(
                monitor='val_R2Score',  # Monitor the validation loss
                patience=20,         # Number of epochs with no improvement after which training will be stopped
                restore_best_weights=True,  # Restore model weights from the epoch with the best validation loss
                verbose=1            # Verbosity mode, 1 = progress messages
            )

qpann.compile(optimizer, loss=tf.keras.losses.MSE, metrics=['R2Score', 'mae'])
qpann.summary()
# Fit the model using training data and include validation data
history = qpann.fit(
    x, 
    simulated_probabilities, 
    epochs=100,
    #validation_data=(x_new['validate'], y_split[sfs][nm_ind][num_shots]['validate']),  # Include validation data
    callbacks=[early_stopping], 
    verbose=1, 
    batch_size=100, 
    validation_batch_size=100,
    shuffle=True
)

fig = plt.figure(figsize=(7,4))
plt.plot(history.history['loss'], label='loss')